# LeetCode 859 — Buddy Strings bằng Python

**Link đề:** https://leetcode.com/problems/buddy-strings/
**Độ khó:** Easy · **Chủ đề:** Hash Table, String

```python
class Solution:
    def buddyStrings(self, s: str, goal: str) -> bool:
        ...
```

## Mục tiêu học
1. Hiểu chính xác yêu cầu: **bắt buộc** hoán đổi **đúng một lần** hai vị trí **khác nhau** `i != j`.
2. Đi từ brute force thử mọi cặp hoán đổi `O(n³)` → **chỉ nhìn các vị trí lệch** `O(n)`.
3. Học cách **phân loại trường hợp (case analysis)**: độ dài khác nhau / hai chuỗi bằng nhau / khác nhau.
4. Dùng `set` để phát hiện **ký tự lặp** — chìa khóa cho ca `s == goal`.
5. Test các bẫy: `"ab"` vs `"ab"`, `"aa"` vs `"aa"`, lệch 1 vị trí, lệch 3 vị trí, khác độ dài.

## 1. Đề bài và ví dụ

Cho hai chuỗi `s` và `goal`. Trả về `True` nếu có thể **hoán đổi hai chữ cái** trong `s` (một lần, tại hai chỉ số `i != j`) để được đúng `goal`.

Ví dụ hoán đổi chỉ số 0 và 2 trong `"abcd"` cho `"cbad"`.

| s | goal | Kết quả | Vì sao |
|---|---|---|---|
| `"ab"` | `"ba"` | `True` | đổi `s[0]`, `s[1]` |
| `"ab"` | `"ab"` | `False` | **bắt buộc** đổi; đổi duy nhất được `"ba"` ≠ goal |
| `"aa"` | `"aa"` | `True` | đổi hai chữ `a` cho nhau, chuỗi không đổi |

**Constraints:** `1 <= len(s), len(goal) <= 2·10^4`; chỉ chữ thường.

> Bẫy lớn nhất: ví dụ 2 vs ví dụ 3 — khi `s == goal` thì phải hoán đổi "vô hại", chỉ được khi có **hai ký tự giống nhau**.

In [1]:
s = "abcd"
lst = list(s)                  # chuỗi bất biến -> đổi sang list để hoán đổi
lst[0], lst[2] = lst[2], lst[0]
print("".join(lst))

cbad


## 2. Nền tảng Python cần dùng

### 2.1. Chuỗi bất biến → muốn hoán đổi phải dùng `list`

`s[0] = "x"` báo lỗi. Ta đổi `list(s)`, hoán đổi `a[i], a[j] = a[j], a[i]`, rồi `"".join(a)`.

In [2]:
s = "ab"
try:
    s[0] = "x"
except TypeError as e:
    print("TypeError:", e)
a = list(s)
a[0], a[1] = a[1], a[0]
print(a, "->", "".join(a))

TypeError: 'str' object does not support item assignment
['b', 'a'] -> ba


### 2.2. `zip(s, goal)` — duyệt song song hai chuỗi

`zip` ghép phần tử cùng vị trí thành cặp. Kết hợp `enumerate` để có chỉ số.

In [3]:
for i, (x, y) in enumerate(zip("abcd", "cbad")):
    print(i, x, y, "LỆCH" if x != y else "")

0 a c LỆCH
1 b b 
2 c a LỆCH
3 d d 


### 2.3. `set` — phát hiện ký tự lặp

`set(s)` giữ mỗi ký tự **một lần**. Nếu `len(set(s)) < len(s)` thì có ít nhất một ký tự xuất hiện ≥ 2 lần.

In [4]:
for s in ["abc", "aab", "a", "abca"]:
    print(f"{s!r}: set={sorted(set(s))}, có lặp? {len(set(s)) < len(s)}")

'abc': set=['a', 'b', 'c'], có lặp? False
'aab': set=['a', 'b'], có lặp? True
'a': set=['a'], có lặp? False
'abca': set=['a', 'b', 'c'], có lặp? True


### 2.4. List comprehension có điều kiện — gom chỉ số lệch

In [5]:
s, goal = "abcd", "cbad"
diff = [i for i in range(len(s)) if s[i] != goal[i]]
print("các vị trí lệch:", diff)

các vị trí lệch: [0, 2]


## 3. Cách 1 — Brute force: thử mọi cặp `(i, j)`

Với mỗi cặp `i < j`, tạo bản sao, hoán đổi, so với `goal`. Có `n(n-1)/2` cặp, mỗi lần tạo/so chuỗi `O(n)` ⇒ `O(n³)`. Với `n = 2·10^4` là ~`4·10^12` bước — không thể. Nhưng là **oracle** tốt cho chuỗi ngắn.

In [6]:
def buddy_brute(s: str, goal: str) -> bool:
    if len(s) != len(goal):
        return False
    n = len(s)
    for i in range(n):
        for j in range(i + 1, n):
            a = list(s)
            a[i], a[j] = a[j], a[i]
            if "".join(a) == goal:
                return True
    return False

print(buddy_brute("ab", "ba"), buddy_brute("ab", "ab"), buddy_brute("aa", "aa"))

True False True


### Nút thắt

Hầu hết các cặp `(i, j)` là vô vọng: nếu hoán đổi `i`, `j` thì **mọi vị trí khác** giữ nguyên, nên chúng phải **đã khớp** `goal` từ trước. Vậy chỉ các vị trí **lệch** mới quyết định kết quả.

## 4. Insight — Đếm các vị trí lệch

Gọi `diff` = danh sách chỉ số `k` mà `s[k] != goal[k]`. Một lần hoán đổi chỉ thay đổi đúng 2 vị trí, nên:

| Trường hợp | Điều kiện để `True` |
|---|---|
| `len(s) != len(goal)` | không bao giờ |
| `len(diff) == 0` (`s == goal`) | cần hoán đổi "vô hại": có **ký tự lặp** trong `s` |
| `len(diff) == 2`, `diff = [i, j]` | đổi chéo khớp: `s[i] == goal[j]` **và** `s[j] == goal[i]` |
| `len(diff) == 1` hoặc `>= 3` | không thể (một lần đổi sửa tối đa 2 vị trí; sửa đúng 1 vị trí là không thể vì đổi luôn tác động 2 vị trí) |

In [7]:
for s, goal in [("ab", "ba"), ("ab", "ab"), ("aa", "aa"), ("abcd", "abdc"), ("abc", "abd"), ("abcd", "badc")]:
    diff = [i for i in range(len(s)) if s[i] != goal[i]]
    print(f"{s!r:7} vs {goal!r:7} -> diff={diff}")

'ab'    vs 'ba'    -> diff=[0, 1]
'ab'    vs 'ab'    -> diff=[]
'aa'    vs 'aa'    -> diff=[]
'abcd'  vs 'abdc'  -> diff=[2, 3]
'abc'   vs 'abd'   -> diff=[2]
'abcd'  vs 'badc'  -> diff=[0, 1, 2, 3]


## 5. Lời giải chính — phân loại trường hợp (từng đoạn rồi ghép)

**Đoạn 1 — khác độ dài:** `if len(s) != len(goal): return False`.

**Đoạn 2 — bằng nhau:** `if s == goal: return len(set(s)) < len(s)`.

**Đoạn 3 — gom vị trí lệch:** `diff = [i for i ... if s[i] != goal[i]]`.

**Đoạn 4 — kiểm tra đúng 2 vị trí, khớp chéo:**
```python
return len(diff) == 2 and s[diff[0]] == goal[diff[1]] and s[diff[1]] == goal[diff[0]]
```

In [8]:
def buddy_cases(s: str, goal: str) -> bool:
    if len(s) != len(goal):
        return False
    if s == goal:
        return len(set(s)) < len(s)
    diff = [i for i in range(len(s)) if s[i] != goal[i]]
    return (len(diff) == 2
            and s[diff[0]] == goal[diff[1]]
            and s[diff[1]] == goal[diff[0]])

print(buddy_cases("ab", "ba"), buddy_cases("ab", "ab"), buddy_cases("aa", "aa"))

True False True


## 6. TRACE từng bước

### 6.1. Bảng trace cho `s = "abcaa"`, `goal = "abcbb"` — lệch 2 vị trí nhưng **không** khớp chéo

| k | s[k] | goal[k] | lệch? | diff |
|---|---|---|---|---|
| 0 | a | a | | [] |
| 1 | b | b | | [] |
| 2 | c | c | | [] |
| 3 | a | b | ✓ | [3] |
| 4 | a | b | ✓ | [3, 4] |

`len(diff) == 2`, nhưng `s[3] = 'a'` ≠ `goal[4] = 'b'` ⇒ `False`. (Đổi `a` với `a` vẫn ra `"abcaa"`.)

In [9]:
def trace_buddy(s: str, goal: str) -> bool:
    print(f"s={s!r}, goal={goal!r}")
    if len(s) != len(goal):
        print("  khác độ dài -> False"); return False
    if s == goal:
        has_dup = len(set(s)) < len(s)
        print(f"  s == goal; set(s)={sorted(set(s))}; có ký tự lặp? {has_dup} -> {has_dup}")
        return has_dup
    diff = []
    for k in range(len(s)):
        mark = ""
        if s[k] != goal[k]:
            diff.append(k); mark = "LỆCH"
        print(f"  k={k}: s[k]={s[k]} goal[k]={goal[k]} {mark:4} diff={diff}")
    if len(diff) != 2:
        print(f"  có {len(diff)} vị trí lệch (cần đúng 2) -> False"); return False
    i, j = diff
    ok = s[i] == goal[j] and s[j] == goal[i]
    print(f"  khớp chéo? s[{i}]={s[i]} vs goal[{j}]={goal[j]}, s[{j}]={s[j]} vs goal[{i}]={goal[i]} -> {ok}")
    return ok

trace_buddy("abcaa", "abcbb");

s='abcaa', goal='abcbb'
  k=0: s[k]=a goal[k]=a      diff=[]
  k=1: s[k]=b goal[k]=b      diff=[]
  k=2: s[k]=c goal[k]=c      diff=[]
  k=3: s[k]=a goal[k]=b LỆCH diff=[3]
  k=4: s[k]=a goal[k]=b LỆCH diff=[3, 4]
  khớp chéo? s[3]=a vs goal[4]=b, s[4]=a vs goal[3]=b -> False


### 6.2. Trace 3 ví dụ đề và một ca lệch 3 vị trí

In [10]:
for s, goal in [("ab", "ba"), ("ab", "ab"), ("aa", "aa"), ("abc", "bca")]:
    trace_buddy(s, goal)
    print()

s='ab', goal='ba'
  k=0: s[k]=a goal[k]=b LỆCH diff=[0]
  k=1: s[k]=b goal[k]=a LỆCH diff=[0, 1]
  khớp chéo? s[0]=a vs goal[1]=a, s[1]=b vs goal[0]=b -> True

s='ab', goal='ab'
  s == goal; set(s)=['a', 'b']; có ký tự lặp? False -> False

s='aa', goal='aa'
  s == goal; set(s)=['a']; có ký tự lặp? True -> True

s='abc', goal='bca'
  k=0: s[k]=a goal[k]=b LỆCH diff=[0]
  k=1: s[k]=b goal[k]=c LỆCH diff=[0, 1]
  k=2: s[k]=c goal[k]=a LỆCH diff=[0, 1, 2]
  có 3 vị trí lệch (cần đúng 2) -> False



## 7. Cách 3 — Một lượt, dừng sớm (không lưu cả danh sách)

Tối ưu nhỏ: khi đang gom `diff`, nếu đã có **3** vị trí lệch thì dừng ngay — chắc chắn `False`. Bộ nhớ chỉ còn tối đa 2 chỉ số (ngoài `set` cho ca bằng nhau).

In [11]:
def buddy_early_stop(s: str, goal: str) -> bool:
    if len(s) != len(goal):
        return False
    first = second = -1
    for k in range(len(s)):
        if s[k] != goal[k]:
            if first == -1:
                first = k
            elif second == -1:
                second = k
            else:
                return False               # lệch thứ 3 -> dừng sớm
    if first == -1:                        # không lệch: s == goal
        return len(set(s)) < len(s)
    if second == -1:                       # chỉ lệch 1
        return False
    return s[first] == goal[second] and s[second] == goal[first]

print(buddy_early_stop("ab", "ba"), buddy_early_stop("ab", "ab"), buddy_early_stop("aa", "aa"))

True False True


`first = second = -1` là **gán chuỗi**: cả hai biến cùng nhận `-1`. `-1` dùng làm giá trị "chưa có".

## 8. Code nộp LeetCode

In [12]:
class Solution:
    def buddyStrings(self, s: str, goal: str) -> bool:
        if len(s) != len(goal):
            return False
        if s == goal:
            return len(set(s)) < len(s)
        diff = [i for i in range(len(s)) if s[i] != goal[i]]
        return (len(diff) == 2
                and s[diff[0]] == goal[diff[1]]
                and s[diff[1]] == goal[diff[0]])

sol = Solution()
assert sol.buddyStrings("ab", "ba") is True
assert sol.buddyStrings("ab", "ab") is False
assert sol.buddyStrings("aa", "aa") is True
print("Solution PASS 3 ví dụ đề")

Solution PASS 3 ví dụ đề


## 9. Vì sao đúng? Độ phức tạp

Giả sử `len(s) == len(goal) == n`. Hoán đổi `i`, `j` (`i != j`) tạo chuỗi `s'` khác `s` **chỉ** ở vị trí `i` và `j`: `s'[i] = s[j]`, `s'[j] = s[i]`.

- **`s' == goal` đòi hỏi:** mọi `k ∉ {i, j}` có `s[k] == goal[k]` ⇒ `diff ⊆ {i, j}`, tức `len(diff) <= 2`.
- **`len(diff) == 2`:** buộc `{i, j} = diff` và cần `s[j] == goal[i]`, `s[i] == goal[j]` — đúng điều kiện khớp chéo.
- **`len(diff) == 1`, `diff = [k]`:** nếu `k ∈ {i, j}`, giả sử `k = i`, thì `j ∉ diff` nên `s[j] == goal[j]`; mà ta cần `s'[j] = s[i] == goal[j] = s[j]` ⇒ `s[i] == s[j]` ⇒ `s'[i] = s[j] = s[i] != goal[i]` — mâu thuẫn. Nếu `k ∉ {i, j}`, vị trí `k` vẫn lệch. ⇒ không thể.
- **`len(diff) == 0`:** cần `s'[i] = s[j] == goal[i] = s[i]` ⇒ `s[i] == s[j]` — cần hai vị trí cùng ký tự ⇔ có ký tự lặp.

**Độ phức tạp:** `n = len(s)`.
- Time `O(n)` — một lượt so sánh + `set(s)`.
- Space: `O(1)` phụ cho cách dừng sớm (bảng chữ cố định 26 nên `set(s)` tối đa 26 phần tử); cách gom `diff` đầy đủ có thể `O(n)` khi lệch nhiều.

| Cách | Time | Space phụ |
|---|---|---|
| 1. Brute force mọi cặp | `O(n³)` | `O(n)` |
| 2. Phân loại + `diff` | `O(n)` | `O(n)` tệ nhất |
| 3. Một lượt dừng sớm | `O(n)` | `O(1)` (≤ 26 ký tự trong set) |
| 4. Counter + diff (mục 10) | `O(n)` | `O(1)` (≤ 26 khóa) |

## 10. Cách 4 — Dùng `Counter` (góc nhìn "đếm tần suất")

`collections.Counter(s)` đếm số lần xuất hiện của mỗi ký tự. Hai điều kiện cần:
- `Counter(s) == Counter(goal)` (cùng "túi" ký tự — hoán đổi không đổi tần suất),
- số vị trí lệch là `2`, hoặc là `0` và có ký tự xuất hiện ≥ 2 lần.

Với `len(diff) == 2` và cùng túi ký tự thì khớp chéo **tự động** đúng (hai cặp lệch phải "trao đổi" cho nhau).

In [13]:
from collections import Counter
print(Counter("aabc"), Counter("aabc").most_common(1))

def buddy_counter(s: str, goal: str) -> bool:
    if len(s) != len(goal) or Counter(s) != Counter(goal):
        return False
    mismatches = sum(1 for x, y in zip(s, goal) if x != y)
    if mismatches == 0:
        return max(Counter(s).values()) >= 2
    return mismatches == 2

print(buddy_counter("ab", "ba"), buddy_counter("ab", "ab"), buddy_counter("aa", "aa"), buddy_counter("abcaa", "abcbb"))

Counter({'a': 2, 'b': 1, 'c': 1}) [('a', 2)]
True False True False


**Vì sao khớp chéo tự động?** Nếu lệch đúng ở `i`, `j` và hai túi ký tự bằng nhau, thì sau khi bỏ phần khớp, túi `{s[i], s[j]}` phải bằng túi `{goal[i], goal[j]}`. Vì `s[i] != goal[i]`, buộc `s[i] == goal[j]`, rồi `s[j] == goal[i]`.

## 11. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu

Nhóm test:
1. Ba ví dụ đề.
2. Khác độ dài.
3. Lệch đúng 1 vị trí; lệch 3 vị trí.
4. Lệch 2 vị trí nhưng không khớp chéo (`"abcaa"`/`"abcbb"`).
5. `s == goal` có / không có ký tự lặp, độ dài 1.

In [14]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
check("đề 1", sol.buddyStrings("ab", "ba"), True)
check("đề 2", sol.buddyStrings("ab", "ab"), False)
check("đề 3", sol.buddyStrings("aa", "aa"), True)
check("khác độ dài", sol.buddyStrings("ab", "abc"), False)
check("lệch 1", sol.buddyStrings("abc", "abd"), False)
check("lệch 3", sol.buddyStrings("abc", "bca"), False)
check("lệch 2 không chéo", sol.buddyStrings("abcaa", "abcbb"), False)
check("lệch 2 xa nhau", sol.buddyStrings("abcdefg", "gbcdefa"), True)
check("bằng nhau, có lặp", sol.buddyStrings("abab", "abab"), True)
check("bằng nhau, không lặp", sol.buddyStrings("abcd", "abcd"), False)
check("độ dài 1", sol.buddyStrings("a", "a"), False)

PASS đề 1: True
PASS đề 2: False
PASS đề 3: True
PASS khác độ dài: False
PASS lệch 1: False
PASS lệch 3: False
PASS lệch 2 không chéo: False
PASS lệch 2 xa nhau: True
PASS bằng nhau, có lặp: True
PASS bằng nhau, không lặp: False
PASS độ dài 1: False


### 11.1. Đối chiếu với brute force trên chuỗi ngẫu nhiên (có seed)

Sinh `goal` bằng cách **hoán đổi ngẫu nhiên** từ `s` một nửa số lần (để có nhiều ca `True`), nửa còn lại sinh ngẫu nhiên.

In [15]:
import random
random.seed(859)
impls = [buddy_cases, buddy_early_stop, buddy_counter, Solution().buddyStrings]
true_count = 0
for _ in range(2000):
    n = random.randint(1, 6)
    s = "".join(random.choice("abc") for _ in range(n))
    if random.random() < 0.5 and n >= 2:
        a = list(s); i, j = random.sample(range(n), 2)
        a[i], a[j] = a[j], a[i]
        goal = "".join(a)
    else:
        goal = "".join(random.choice("abc") for _ in range(random.randint(1, 6)))
    expected = buddy_brute(s, goal)
    true_count += expected
    for f in impls:
        assert f(s, goal) == expected, (s, goal, f)
print(f"2000 cặp: 4 cài đặt khớp brute force ({true_count} ca True)")

2000 cặp: 4 cài đặt khớp brute force (859 ca True)


`random.sample(range(n), 2)` chọn **2 số khác nhau** — bảo đảm `i != j`. `random.random()` trả số thực trong `[0, 1)`.

## 12. Lỗi người mới hay gặp

### Lỗi 1: `s == goal` thì trả luôn `True`

Quên rằng **bắt buộc** phải hoán đổi.

In [16]:
def wrong_equal(s, goal):
    if s == goal:
        return True                          # SAI
    return Solution().buddyStrings(s, goal)

print("sai :", wrong_equal("ab", "ab"), "| đúng:", Solution().buddyStrings("ab", "ab"))

sai : True | đúng: False


### Lỗi 2: chỉ đếm số vị trí lệch == 2, quên kiểm tra khớp chéo

In [17]:
def wrong_no_cross(s, goal):
    if len(s) != len(goal):
        return False
    if s == goal:
        return len(set(s)) < len(s)
    return sum(x != y for x, y in zip(s, goal)) == 2

print("sai :", wrong_no_cross("abcaa", "abcbb"), "| đúng:", Solution().buddyStrings("abcaa", "abcbb"))

sai : True | đúng: False


`sum(x != y for ...)`: `True` được tính là `1`, `False` là `0` — cách đếm nhanh số điều kiện đúng.

### Lỗi 3: quên kiểm tra độ dài, `zip` âm thầm cắt bớt

`zip("ab", "bac")` chỉ ghép 2 cặp → so sánh thiếu ký tự `'c'` của `goal`.

In [18]:
print(list(zip("ab", "bac")))
def wrong_length(s, goal):
    diff = [i for i, (x, y) in enumerate(zip(s, goal)) if x != y]
    return len(diff) == 2 and s[diff[0]] == goal[diff[1]] and s[diff[1]] == goal[diff[0]]

print("sai :", wrong_length("ab", "bac"), "| đúng:", Solution().buddyStrings("ab", "bac"))

[('a', 'b'), ('b', 'a')]
sai : True | đúng: False


### Lỗi 4: kiểm tra ký tự lặp bằng `len(set(s)) < 26` hoặc "độ dài ≥ 2"

Có ký tự lặp ⇔ `len(set(s)) < len(s)`, **không** phải `< 26`. Ví dụ `"ab"` có `len(set) = 2 < 26` nhưng không có ký tự lặp.

In [19]:
s = "ab"
print("sai :", len(set(s)) < 26, "| đúng:", len(set(s)) < len(s))

sai : True | đúng: False


## 13. Mẹo nhận dạng pattern

- "Có thể biến A thành B bằng **đúng một** thao tác (hoán đổi / thay / xóa)?" → so sánh song song, **đếm vị trí lệch**; thao tác ảnh hưởng `k` vị trí thì số lệch phải `<= k`.
- Luôn tách ca "đã bằng nhau sẵn" — thường có điều kiện riêng (ở đây là ký tự lặp).
- Tần suất ký tự (`Counter`) là bất biến dưới phép hoán đổi — điều kiện **cần** hữu ích.
- Bài cùng họ: *One Edit Distance (161)*, *Check if One String Swap Can Make Strings Equal (1790)* (giống 859 nhưng **cho phép** không đổi), *Determine if Two Strings Are Close (1657)*.

## 14. Biến thể

### 14.1. LeetCode 1790 — "**tối đa** một lần hoán đổi" (được phép không đổi)

Chỉ khác ca `s == goal`: trả luôn `True`.

In [20]:
def are_almost_equal(s1: str, s2: str) -> bool:
    diff = [i for i in range(len(s1)) if s1[i] != s2[i]]
    if not diff:
        return True
    return len(diff) == 2 and s1[diff[0]] == s2[diff[1]] and s1[diff[1]] == s2[diff[0]]

print(are_almost_equal("bank", "kanb"), are_almost_equal("attack", "defend"), are_almost_equal("kelb", "kelb"))

True False True


### 14.2. Trả về **cặp chỉ số** cần hoán đổi (hoặc `None`)

In [21]:
def swap_pair(s: str, goal: str):
    if len(s) != len(goal):
        return None
    if s == goal:
        seen = {}
        for i, ch in enumerate(s):
            if ch in seen:
                return (seen[ch], i)
            seen[ch] = i
        return None
    diff = [i for i in range(len(s)) if s[i] != goal[i]]
    if len(diff) == 2 and s[diff[0]] == goal[diff[1]] and s[diff[1]] == goal[diff[0]]:
        return tuple(diff)
    return None

print(swap_pair("abcd", "cbad"), swap_pair("xyzy", "xyzy"), swap_pair("ab", "ab"))

(0, 2) (1, 3) None


### 14.3. Mảng số: có thể sắp xếp tăng dần bằng **một** lần hoán đổi?

So sánh với `sorted(nums)` — cùng ý tưởng "đếm vị trí lệch".

In [22]:
def sortable_by_one_swap(nums: list[int]) -> bool:
    target = sorted(nums)
    diff = [i for i in range(len(nums)) if nums[i] != target[i]]
    return len(diff) == 0 or (len(diff) == 2 and nums[diff[0]] == target[diff[1]] and nums[diff[1]] == target[diff[0]])

print(sortable_by_one_swap([1, 5, 3, 4, 2]), sortable_by_one_swap([3, 1, 2]))

True False


## 15. Bài tập (tự làm trước, đáp án ở cuối)

### Bài 1 — Tính tay
Với `s = "converse"`, `goal = "conserve"`: liệt kê `diff`, kiểm tra khớp chéo, kết luận.

### Bài 2 — Dự đoán output
Không chạy, đoán: `len(set("hello"))`, `list(zip("ab", "xyz"))`, `sum([True, False, True])`.

### Bài 3 — Điền TODO

In [23]:
def ex3_buddy(s: str, goal: str) -> bool:
    # TODO 1: khác độ dài -> False
    # TODO 2: s == goal -> có ký tự lặp?
    # TODO 3: gom diff, kiểm tra đúng 2 vị trí và khớp chéo
    pass

# assert ex3_buddy("ab", "ba") is True
# assert ex3_buddy("ab", "ab") is False
# assert ex3_buddy("aa", "aa") is True

### Bài 4 — LeetCode 1790 (tối đa một lần hoán đổi)
Tự viết lại không nhìn mục 14.1.

In [24]:
def ex4_almost_equal(s1: str, s2: str) -> bool:
    # TODO
    pass

# assert ex4_almost_equal("bank", "kanb") is True
# assert ex4_almost_equal("kelb", "kelb") is True
# assert ex4_almost_equal("attack", "defend") is False

### Bài 5 — Một lần **thay** ký tự
Trả `True` nếu `s` và `goal` cùng độ dài và khác nhau **đúng** một vị trí.

In [25]:
def ex5_one_replace(s: str, goal: str) -> bool:
    # TODO
    pass

# assert ex5_one_replace("cat", "cut") is True
# assert ex5_one_replace("cat", "cat") is False
# assert ex5_one_replace("cat", "dog") is False

### Bài 6 — Mini challenge: số lần hoán đổi tối thiểu (hai chữ)
Cho `s`, `goal` cùng độ dài, chỉ gồm `'x'` và `'y'`, với cùng số chữ `x`. Mỗi bước hoán đổi hai vị trí **trong `s`**. Tìm số bước tối thiểu để `s == goal`.

In [26]:
def ex6_min_swaps_xy(s: str, goal: str) -> int:
    # TODO
    pass

# assert ex6_min_swaps_xy("xxyy", "yyxx") == 2
# assert ex6_min_swaps_xy("xyxy", "xyxy") == 0

## 16. Gợi ý 3 mức

**Mức 1 — hướng tư duy**
- Bài 1: so từng vị trí.
- Bài 5: đếm số vị trí lệch.
- Bài 6: mỗi lần hoán đổi một vị trí "x cần thành y" với một vị trí "y cần thành x" sửa được **2** vị trí lệch cùng lúc.

**Mức 2 — cú pháp**
- Bài 3: như mục 5.
- Bài 5: `len(s) == len(goal) and sum(a != b for a, b in zip(s, goal)) == 1`.

**Mức 3 — gần code**
- Bài 6: số vị trí lệch `d` luôn chẵn (cùng số `x`); đáp án là `d // 2`.

## 17. Đáp án tham khảo

**Bài 1:** `c o n v e r s e` vs `c o n s e r v e` ⇒ `diff = [3, 6]`. Khớp chéo: `s[3] = 'v' == goal[6] = 'v'` và `s[6] = 's' == goal[3] = 's'` ⇒ `True`.

**Bài 2:** `4`, `[('a', 'x'), ('b', 'y')]`, `2`.

In [27]:
print(len(set("hello")), list(zip("ab", "xyz")), sum([True, False, True]))
print("diff:", [i for i in range(8) if "converse"[i] != "conserve"[i]])
assert Solution().buddyStrings("converse", "conserve") is True

def ans3_buddy(s, goal):
    if len(s) != len(goal):
        return False
    if s == goal:
        return len(set(s)) < len(s)
    diff = [i for i in range(len(s)) if s[i] != goal[i]]
    return len(diff) == 2 and s[diff[0]] == goal[diff[1]] and s[diff[1]] == goal[diff[0]]

def ans4_almost_equal(s1, s2):
    diff = [i for i in range(len(s1)) if s1[i] != s2[i]]
    return not diff or (len(diff) == 2 and s1[diff[0]] == s2[diff[1]] and s1[diff[1]] == s2[diff[0]])

def ans5_one_replace(s, goal):
    return len(s) == len(goal) and sum(a != b for a, b in zip(s, goal)) == 1

def ans6_min_swaps_xy(s, goal):
    d = sum(a != b for a, b in zip(s, goal))
    return d // 2

assert ans3_buddy("ab", "ba") is True and ans3_buddy("ab", "ab") is False and ans3_buddy("aa", "aa") is True
assert ans4_almost_equal("bank", "kanb") is True
assert ans4_almost_equal("kelb", "kelb") is True
assert ans4_almost_equal("attack", "defend") is False
assert ans5_one_replace("cat", "cut") is True
assert ans5_one_replace("cat", "cat") is False
assert ans5_one_replace("cat", "dog") is False
assert ans6_min_swaps_xy("xxyy", "yyxx") == 2
assert ans6_min_swaps_xy("xyxy", "xyxy") == 0
print("Đáp án bài 1-6 PASS")

4 [('a', 'x'), ('b', 'y')] 2
diff: [3, 6]
Đáp án bài 1-6 PASS


## 18. Tổng kết — template nhớ nhanh và checklist submit

```python
if len(s) != len(goal): return False
if s == goal:           return len(set(s)) < len(s)      # cần ký tự lặp
diff = [i for i in range(len(s)) if s[i] != goal[i]]
return len(diff) == 2 and s[diff[0]] == goal[diff[1]] and s[diff[1]] == goal[diff[0]]
```

**Câu thần chú:** *"Một lần đổi sửa đúng 2 chỗ: lệch 0 → cần ký tự lặp; lệch 2 → phải khớp chéo; còn lại → False."*

**Checklist trước khi submit:**
- [ ] Kiểm tra độ dài trước (đừng để `zip` cắt bớt)?
- [ ] Ca `s == goal` dùng `len(set(s)) < len(s)`, **không** trả luôn `True`?
- [ ] Ca lệch 2 có kiểm tra **khớp chéo** cả hai chiều?
- [ ] Ca lệch 1 hoặc ≥ 3 trả `False`?
- [ ] Đã test `"ab"/"ab"`, `"aa"/"aa"`, `"abcaa"/"abcbb"`?